# Boltz-2 virtual screening on a Colab GPU

One protein target, many ligands, co-folded and scored for binding by
[Boltz-2](https://github.com/jwohlwend/boltz) — but on **this notebook's GPU**
instead of your laptop's.

The page normally runs entirely in your own browser, which caps the target at
roughly 2,000 residues: one pair tensor is `residues² × 128 × 4` bytes and a
WebGPU binding cannot exceed `maxStorageBufferBindingSize`. A Colab A100
reports a far larger binding, so **bigger targets fold here** — the link this
cell prints says the ceiling it measured.

**Run the cell, then click the link.** Keep the notebook running while you use
the page; Runtime ▸ Disconnect hands the machine back.

Choose a GPU first: Runtime ▸ Change runtime type ▸ T4, L4 or A100. Without
one the cell still works and says loudly that it is the CPU renderer.

Repository: https://github.com/pablo-arantes/boltz2-webgpu ·
the arrangement is ported from [LocalFold](https://github.com/sokrypton/localfold)
(Beerware, Sergey Ovchinnikov).


In [ ]:
#@title Boltz-2 virtual screening, on this Colab GPU { display-mode: "form" }
#
# Run this cell, then click the link it prints. The page is this notebook's
# machine: the screen runs on the Colab GPU, so a target far larger than a
# laptop can hold will fold - an A100 reports a much bigger WebGPU binding
# limit, and the link box prints the residue ceiling it measured.
#
# Keep this notebook running while you use the page. Runtime > Disconnect, or
# the page's own Disconnect, hands the machine back.
import json, os, queue, subprocess, sys, threading, time

BRANCH, PORT, REPO = 'main', 8710, '/content/boltz2-webgpu'
REPO_URL = 'https://github.com/pablo-arantes/boltz2-webgpu'

# The setup: a Vulkan-capable driver, a headless Chrome, and this repository.
# Each step checks the disk first, so re-running the cell is nearly free. The
# unusual choices here - a zipped Chrome rather than the .deb, the driver
# unpacked rather than installed - are LocalFold's, and its reasoning is kept
# with them: docs/COLAB.md records what was ported and why.
SETUP = r"""
set -e
say() { echo "[$(( $(date +%s) - T0 ))s] $*"; }
T0=$(date +%s)

# What is missing? Each probe is an `if`: under `set -e` a bare `[ -e X ] && f=0`
# would end the script when the test fails, which is the case we are detecting.
need_icd=1
if [ -e /usr/share/vulkan/icd.d/nvidia_icd.json ]; then need_icd=0; fi
need_chrome=1
if command -v google-chrome > /dev/null 2>&1; then need_chrome=0; fi
need_loader=1
if ldconfig -p 2>/dev/null | grep -q libvulkan.so.1; then need_loader=0; fi
say "icd=$need_icd chrome=$need_chrome loader=$need_loader"

# Chrome, as a zip rather than a .deb: same binary, no package manager.
if [ "$need_chrome" = 1 ]; then
  ( cd /content \
    && wget -q https://storage.googleapis.com/chrome-for-testing-public/153.0.8010.52/linux64/chrome-headless-shell-linux64.zip \
    && unzip -q -o chrome-headless-shell-linux64.zip -d /opt \
    && ln -sf /opt/chrome-headless-shell-linux64/chrome-headless-shell /usr/local/bin/google-chrome ) &
  chrome_download=$!
fi

# The repository. Started first and waited for last: it is git's network while
# the rest is dpkg's lock, so it costs nothing to run underneath them.
if [ -d REPO_PATH ]; then
  ( git -C REPO_PATH fetch -q origin BRANCH && git -C REPO_PATH checkout -q FETCH_HEAD ) &
else
  ( git clone -q --depth 1 --branch BRANCH REPO_URL REPO_PATH ) &
fi
repo_clone=$!

# The Vulkan loader, and the four libraries a zipped Chrome asks for that this
# image lacks. The `t64` spelling is Ubuntu 24.04's, the bare one 22.04's.
if [ "$need_loader" = 1 ] || [ "$need_icd" = 1 ] || [ "$need_chrome" = 1 ]; then
  APT="apt-get -qq -o Dpkg::Use-Pty=0 --no-install-recommends"
  X11="libatk1.0-0t64 libatk-bridge2.0-0t64 libatspi2.0-0t64 libxcomposite1"
  X11_OLD="libatk1.0-0 libatk-bridge2.0-0 libatspi2.0-0 libxcomposite1"
  ( $APT install -y libvulkan1 $X11 > /dev/null 2>&1 \
    || $APT install -y libvulkan1 $X11_OLD > /dev/null 2>&1 \
    || { apt-get -qq update > /dev/null 2>&1
         $APT install -y libvulkan1 $X11 > /dev/null 2>&1 \
           || $APT install -y libvulkan1 $X11_OLD > /dev/null 2>&1; } ) &
  small_libs=$!
fi

# Colab already has the NVIDIA userspace; what it lacks is a libGLX_nvidia
# carrying the Vulkan entry point, without which WebGPU finds no adapter. Only
# that package is unpacked, into `/` so the linker mixes it with Colab's own
# libraries at the kernel module's version.
if [ "$need_icd" = 1 ]; then
  D=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | cut -d. -f1)
  if [ -n "$D" ]; then
    ( cd /content && apt-get -qq download libnvidia-gl-$D > /dev/null 2>&1 \
      && dpkg-deb -x libnvidia-gl-${D}_*.deb / && ldconfig )
    say "driver (unpacked, not installed)"
  fi
fi
# `wait $x` with `x` unset is a bare `wait`, which reaps every job - and a
# second wait on a reaped pid returns 127, which `set -e` turns into an exit
# with no message. On a WARM box the three probes are all 0, so `small_libs` is
# never set, the bare wait collects the clone as well, and `wait $repo_clone`
# below ends the setup at rc 127 with the reason swallowed. Cold boxes passed
# and every RE-run failed - LocalFold measured this on a Colab T4.
if [ -n "$small_libs" ]; then wait $small_libs 2>/dev/null; fi
say "vulkan"

if [ "$need_chrome" = 1 ]; then
  wait $chrome_download
  say "chrome $(google-chrome --version 2>/dev/null || echo 'DID NOT INSTALL')"
fi

wait $repo_clone 2>/dev/null
say "repository"
"""

with open('/content/_boltz2_setup.sh', 'w') as handle:
    handle.write(SETUP.replace('REPO_PATH', REPO).replace('REPO_URL', REPO_URL)
                 .replace('BRANCH', BRANCH))

# The log is kept, not printed: it is only worth reading when there is no link
# at the end, so it is shown in full if the setup fails.
print('setting up…')
_t0 = time.time()
_setup = subprocess.run(['bash', '/content/_boltz2_setup.sh'],
                        capture_output=True, text=True)
if _setup.returncode != 0:
    print(_setup.stdout, _setup.stderr)
    raise SystemExit('the setup failed; the log is above')
_t_setup = time.time() - _t0


def _drain(stream, sink):
    for line in iter(stream.readline, ''):
        sink.put(line.rstrip())


# The service both serves the page and brokers for it, which is what makes one
# cell enough: the link below is bench.html on this machine, so the page and the
# GPU are the same origin. It opens a second, headless copy of that page on the
# card and the two talk through web/colab-bridge.js - yours asks, that one
# screens and pushes back every status line and every scored ligand as it lands.
if not globals().get('_boltz2_service'):
    # The runtime's address is passed explicitly rather than inherited:
    # Disconnect hands the machine back with a POST to $TBE_RUNTIME_ADDR, so a
    # subprocess that merely hopes to inherit it would stop the service and
    # leave the machine assigned.
    _env = dict(os.environ)
    if 'TBE_RUNTIME_ADDR' in os.environ:
        _env['TBE_RUNTIME_ADDR'] = os.environ['TBE_RUNTIME_ADDR']
    _boltz2_service = subprocess.Popen(
        [sys.executable, 'tools/colab_backend.py', '--port', str(PORT)],
        cwd=REPO, env=_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    _lines = queue.Queue()
    threading.Thread(target=_drain, args=(_boltz2_service.stdout, _lines),
                     daemon=True).start()
    TOKEN, ADAPTER, CEILING = None, None, None
    _deadline = time.time() + 420
    while time.time() < _deadline and TOKEN is None:
        try:
            said = _lines.get(timeout=5)
        except queue.Empty:
            continue
        if said.startswith('BACKEND '):
            answer = json.loads(said[len('BACKEND '):])
            TOKEN = answer['token']
            ADAPTER = answer['gpu']
            CEILING = answer.get('residueCeiling')
        elif 'rror' in said:
            print(said)
    if TOKEN is None:
        raise SystemExit('the screening service did not start; run this cell again')

    # Belt to the service's braces: whichever notices first hands the machine
    # back, and the second call is a no-op. A thread rather than a loop, so the
    # cell finishes and the link stays clickable.
    def _release_when_done(service):
        service.wait()
        try:
            from google.colab import runtime as _runtime
            _runtime.unassign()
            print('the Colab runtime has been released')
            return
        except Exception as cause:
            print(f'unassign did not take ({cause}); disconnecting the'
                  ' frontend instead')
        try:
            from google.colab import output as _out
            _out.eval_js('google.colab.kernel.disconnect()', ignore_result=True)
            print('the notebook has been disconnected; the machine idles out')
        except Exception as cause:
            print('the screening service stopped; the runtime is still assigned'
                  f' ({cause}) - use Runtime > Disconnect and delete runtime')

    threading.Thread(target=_release_when_done, args=(_boltz2_service,),
                     daemon=True).start()

# 'nvidia' and an architecture is the card. 'swiftshader' or 'llvmpipe' is the
# CPU wearing its clothes, and every screen after that looks exactly like
# success - so that case is loud and the good case is one line.
_card = '%s %s' % (ADAPTER.get('vendor') or '?', ADAPTER.get('architecture') or '')
_fallback = ('swiftshader' in json.dumps(ADAPTER).lower()
             or 'llvmpipe' in json.dumps(ADAPTER).lower())
if _fallback:
    print('*** %s is the CPU renderer, not the card.' % _card,
          'Runtime > Change runtime type > a GPU, then run this cell again. ***')

from google.colab.output import eval_js
from IPython.display import HTML, display

base = eval_js('google.colab.kernel.proxyPort(%d)' % PORT)
# The base may or may not end in a slash. Without this, '%sbench.html' once
# produced '…prod.colab.devbench.html', which the browser reads as a hostname.
base = base if base.endswith('/') else base + '/'
# `backend=colab` tells the page which machine screens; `t` is the token every
# request carries, in the URL because a page cannot be handed a header by
# whoever opened it.
page = '%sbench.html?backend=colab&t=%s' % (base, TOKEN)
_ceiling = ('up to about %d residues' % CEILING) if CEILING else 'an unknown size'
display(HTML(
    '<div style="font:14px system-ui;padding:14px 16px;border:1px solid #e5e7eb;'
    'border-radius:10px;display:inline-block">'
    '<a href="%s" target="_blank" rel="noopener" '
    'style="font-size:17px;font-weight:600;text-decoration:none">'
    'Boltz-2 virtual screening &rarr;</a>'
    '<div style="color:#6b7280;margin-top:6px">'
    'on this runtime\'s <b>%s</b> &middot; %s<br>'
    'keep this notebook running &middot; the first screen downloads 365 MiB of '
    'weights on the runtime, so it is slower than the ones after it</div>'
    '</div>' % (page, _card, _ceiling)))
